**Feature Selection Method:** CORR   
**Dataset:** QUT-DV25 (Combined)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif

In [2]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [5]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [6]:
gc.collect()
tf.keras.backend.clear_session()


In [7]:
data.head()

,Package_Name,IO_Operations,File_Operations,Network_Operations,Time_Operations,Security_Operations,Process_Operations,Read_Processes,Write_Processes,Read_Data_Transfer,...,Pattern_2,Pattern_3,Pattern_4,Pattern_5,Pattern_6,Pattern_7,Pattern_8,Pattern_9,Pattern_10,Level
0,10Cent10-999.0.4.tar.gz,ioctl,NaN,"getrandom, uname",NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,fstat -> ioctl -> lseek,openat -> fstat -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> openat -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,fstat -> ioctl -> lseek -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,ioctl -> lseek -> lseek -> error=ENOTTY -> no-fd,read -> read -> close -> no-error -> no-fd,1
1,10Cent11-999.0.4.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,"wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,ioctl -> ioctl -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,ioctl -> ioctl -> ioctl -> no-error -> no-fd,fcntl -> fstat -> fcntl -> no-error -> no-fd,1
2,11Cent-999.0.0.tar.gz,"ioctl, poll",pipe2,"getrandom, uname",NaN,NaN,"vfork, wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,newfstatat -> newfstatat -> openat,read -> read -> read,newfstatat -> openat -> fstat,fstat -> ioctl -> lseek,newfstatat -> newfstatat -> newfstatat -> no-e...,newfstatat -> newfstatat -> openat -> no-error...,read -> read -> read -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,1
3,11Cent-999.0.1.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,openat -> fstat -> fcntl,fstat -> fcntl -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,openat -> fstat -> fcntl -> no-error -> no-fd,fstat -> fcntl -> fstat -> no-error -> no-fd,1
4,11Cent-999.0.2.tar.gz,"poll, ioctl",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,read -> poll -> read,poll -> read -> read,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,read -> poll -> read -> error=EAGAIN -> no-fd,newfstatat -> newfstatat -> openat -> no-error...,1


In [8]:
# X = your feature DataFrame, y = target column
X = data.drop(columns=['Level'])
y = data['Level']

# 1. Compute correlation with target
correlation_with_target = X.apply(lambda col: col.corr(y) if np.issubdtype(col.dtype, np.number) else 0)

# 2. Drop features highly correlated with each other (|r| >= 0.5)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Features to drop due to high correlation
to_drop = [column for column in upper.columns if any(upper[column] >= 0.5)]

# 3. Keep features with |r| < 0.5
selected_features = [f for f in X.columns if f not in to_drop]

# 4. If you have feature importance scores (IMS) from ML models, apply threshold
# Example: keep features with IMS > 0.05 in at least one model
# Assuming ims_dict = {'feature_name': max_importance_across_models, ...}
# selected_features = [f for f in selected_features if ims_dict.get(f, 0) > 0.05]

# 5. Final selected features DataFrame
X_selected_df = X[selected_features]

print("Selected features:", list(X_selected_df.columns))
print("Number of features kept:", X_selected_df.shape[1])
print("Number of features dropped:", X.shape[1] - X_selected_df.shape[1])


Selected features: ['Package_Name', 'IO_Operations', 'File_Operations', 'Network_Operations', 'Time_Operations', 'Security_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'Root_DIR_Access', 'Temp_DIR_Access', 'Sys_DIR_Access', 'State_Transition', 'Local_IPs_Access', 'Total_Dependencies', 'Pattern_1', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_7', 'Pattern_8', 'Pattern_9', 'Pattern_10']
Number of features kept: 31
Number of features dropped: 9


In [9]:
selected_features = ['IO_Operations', 'File_Operations', 'Network_Operations', 'Time_Operations', 'Security_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'Root_DIR_Access', 'Temp_DIR_Access', 'Sys_DIR_Access', 'State_Transition', 'Local_IPs_Access', 'Total_Dependencies', 'Pattern_1', 'Pattern_2', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_7', 'Pattern_8', 'Pattern_9', 'Pattern_10']